# Time Pattern Generator

Stage 01 creates the stimulation current waveforms used by the rest of the manuscript pipeline. This notebook illustrates the logic for one CV/frequency pair; the full release data already contain every generated repetition.


## Pipeline Map

For each coefficient of variation (CV) and stimulation frequency, stage 01:

1. Loads the recorded Slack reference waveform for the pair as `rep001_slack`.
2. Draws fixed-count generated pulse times for `rep002` through `rep100`: `frequency * 2 s` events.
3. Uses either a regular train (CV=0) or gamma-distributed inter-pulse intervals (CV>0).
4. Converts each generated pulse time into the sampled biphasic current waveform used by the model.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _bootstrap_src(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        src_dir = candidate / 'src'
        if src_dir.exists() and (candidate / 'data').exists():
            if str(src_dir) not in sys.path:
                sys.path.insert(0, str(src_dir))
            return candidate
    raise RuntimeError('Could not locate repository root.')

REPO_ROOT = _bootstrap_src(Path.cwd())

from utils import require_data_files, data_subdir, ensure_src_on_path, find_repo_root
from figures import set_manuscript_style

REPO_ROOT = find_repo_root(REPO_ROOT)
SRC_DIR = ensure_src_on_path(REPO_ROOT)
require_data_files(
    REPO_ROOT,
    [
        'data/time-patterns/slack',
    ],
    context='01_TimePatterns_Generator',
)


from time_patterns import (
    DEFAULT_CV_VALUES,
    DEFAULT_FREQUENCIES_HZ,
    TimePatternConfig,
    draw_gamma_times,
    generate_biphasic_waveform,
    load_slack_waveform_1ua,
    plot_waveform_construction,
    regular_train_times,
    seed_for_pair,
    slack_path_format,
    verify_time_pattern_outputs,
)

set_manuscript_style(dpi=120)
pd.set_option('display.max_columns', None)


## Configure and Check Data

The data check below runs silently unless a required stage-01 file is missing.


In [ ]:
time_patterns_dir = data_subdir(REPO_ROOT, 'time_patterns')
config = TimePatternConfig(
    cv_vals=DEFAULT_CV_VALUES,
    freq_vals=DEFAULT_FREQUENCIES_HZ,
    pulse_width_s=450e-6,
    stim_duration_s=2.0,
    slack_dir=time_patterns_dir / 'slack',
    output_dir=time_patterns_dir / 'generated',
).resolved()
spec = config.spec

status = verify_time_pattern_outputs(config)
if not status.complete:
    missing = list(status.missing_input_files) + list(status.missing_output_files)
    preview = chr(10).join(str(path) for path in missing[:8])
    raise FileNotFoundError('Stage-01 data are missing. First missing files:' + chr(10) + preview)

example_cv = 0.5
example_freq = 50

## Step 1: Load the Slack Reference

The first repetition in each CV/frequency bundle is the recorded Slack waveform, loaded as-is. It is intentionally not forced to have the exact generated pulse count.


In [ ]:
slack_path = slack_path_format(example_cv, example_freq, config.slack_dir)
slack_waveform = load_slack_waveform_1ua(slack_path, spec)

pd.DataFrame([
    {
        'cv': example_cv,
        'freq_hz': example_freq,
        'rep_label': 'rep001_slack',
        'source_file': str(slack_path.relative_to(REPO_ROOT)),
        'samples': slack_waveform.size,
        'duration_s': slack_waveform.size / spec.fs,
    }
])

## Step 2: Draw Pulse Times

For generated repetitions, CV=0 pulses are exactly periodic. For CV>0, inter-pulse intervals are sampled from a gamma distribution whose coefficient of variation matches the requested CV. These generated repetitions are validated to contain the exact expected pulse count and to fit each full biphasic pulse inside the 0-2 s stimulation window.


In [ ]:
seed = seed_for_pair(config, example_cv, example_freq)
rng = np.random.default_rng(seed)

if example_cv == 0.0:
    pulse_times_s = regular_train_times(
        example_freq,
        spec.stim_dur_s,
        spec=spec,
        first_pulse_sample=1,
    )
else:
    pulse_times_s = draw_gamma_times(
        example_freq,
        example_cv,
        spec.stim_dur_s,
        rng,
        spec,
        first_pulse_sample=1,
    )

pd.DataFrame([
    {
        'seed': seed,
        'n_pulses': pulse_times_s.size,
        'mean_ipi_ms': 1000.0 * np.diff(pulse_times_s).mean(),
        'cv_ipi': np.diff(pulse_times_s).std(ddof=1) / np.diff(pulse_times_s).mean(),
    }
])

## Step 3: Convert Pulse Times to Biphasic Current

Each event time is placed on the 20 kHz sampling grid and rendered as a 4-sample negative phase (200 us), a 1-sample interphase gap (50 us), and a 4-sample positive phase (200 us). Downstream stages multiply this unit-amplitude current waveform by spatial electric-field profiles.


In [ ]:
generated_waveform = generate_biphasic_waveform(pulse_times_s, spec)

pd.DataFrame([
    {
        'samples': generated_waveform.size,
        'nonzero_samples': int(np.count_nonzero(generated_waveform)),
        'pulse_width_us': 1e6 * spec.pulse_samples / spec.fs,
        'min_current_uA': float(generated_waveform.min()),
        'max_current_uA': float(generated_waveform.max()),
    }
])

## Step 4: Inspect the Constructed Waveform

The top panel shows the gamma-distributed event times for one generated time pattern, the middle panel compares the recorded Slack reference with the generated waveform across the full 2 s stimulation window, and the bottom panel zooms into the biphasic pulse structure.


In [ ]:
fig, axes = plot_waveform_construction(
    generated_waveform=generated_waveform,
    pulse_times_s=pulse_times_s,
    spec=spec,
    cv=example_cv,
    freq=example_freq,
    reference_waveform=slack_waveform,
)
display(fig)
plt.close(fig)

## Full Stage Output

The same construction is repeated for all 11 CV values and 5 stimulation frequencies. Each CV/frequency bundle contains `rep001_slack` plus 99 generated repetitions, labeled `rep002_regular` through `rep100_regular` for CV=0 and `rep002_gamma` through `rep100_gamma` for CV>0. Those archived `.npz`, metadata, and manifest files are the stage-01 inputs for stage 02.
